In [ ]:
# Librerías
import requests
import selectolax
from selectolax.parser import HTMLParser
import pandas as pd
import numpy as np
from datetime import datetime
from zoneinfo import ZoneInfo

# Funciones del proyecto
import html_utils
from html_utils import esperar, obtener_max_page_s6_new, save_df_as_csv, extraer_productos_s6_new

##### Categorias

In [ ]:
response = requests.get("https://www.superseis.com.py/default.aspx")
html = response.text
tree = HTMLParser(html)

In [ ]:
data = []

# 🔹 Buscar todas las categorías principales
for lvl1_li in tree.css("li.nav-item.dropdown-categories"):
    lvl1_a = lvl1_li.css_first("a.header-menu, a.dropdown-toggle-categories")
    if not lvl1_a:
        continue
    lvl1_name = lvl1_a.text(strip=True)

    # 🔹 Dentro de cada categoría principal, buscar subcategorías (nivel 2)
    for lvl2_li in lvl1_li.css("li.dropdown-submenu"):
        lvl2_a = lvl2_li.css_first("a.submenu-title[href]")
        if not lvl2_a:
            continue
        lvl2_name = lvl2_a.text(strip=True)
        lvl2_url = lvl2_a.attributes.get("href")

        # 🔹 Dentro de cada subcategoría, buscar sub-subcategorías (nivel 3)
        for lvl3_a in lvl2_li.css("ul.grand-child a[href]"):
            lvl3_name = lvl3_a.text(strip=True)
            lvl3_url = lvl3_a.attributes.get("href")

            data.append({
                "categoria_nivel_1": lvl1_name,
                "categoria_nivel_2": lvl2_name,
                "categoria_nivel_3": lvl3_name,
                "url": lvl3_url,
                "category_slug": f"{lvl1_name}/{lvl2_name}/{lvl3_name}".replace(" ", "_")
            })

In [ ]:
df_categorias = pd.DataFrame(data)
print(f"✅ Total de categorías encontradas: {len(df_categorias)}")

In [20]:
df_categorias.head()

In [21]:
save_df_as_csv(
    dataframe = df_categorias,
    name = 's6_categorias',
    subfolder = 's6/categorias'
)

# Productos

In [22]:
INGESTION_TIME = datetime.now(ZoneInfo("America/Asuncion"))
SUPERMERCADO = "Super Seis"
productos_final = []
selectors = {
    "producto": "div.content",
    "titulo": "div.description h4 a[data-product-name]",
    "precio": "div.price span.price-new",
    "unidad_medida": "span.sale-type-badge",
    "marca": "",  # No disponible actualmente
    }

for _, row in df_categorias.iterrows():
    categoria_url = row["url"]
    category_slug = row["category_slug"]

    print(f"\n🔎 Scrapeando categoría: {category_slug}")
    esperar()
    
    response = requests.get(categoria_url)
    tree = HTMLParser(response.text)
    max_page = obtener_max_page_s6_new(tree)
    print(f"📄 Total de páginas: {max_page}")

    for page in range(1, max_page + 1):
        page_url = f"{categoria_url}?pageindex={page}"
        print(f"➡️ Página {page}: {page_url}")
        esperar()

        resp = requests.get(page_url)
        html_tree = HTMLParser(resp.text)

        contexto = {
            "category_slug": category_slug,
            "ingestion_time": INGESTION_TIME,
            "supermercado": SUPERMERCADO
        }

        productos_pagina = extraer_productos_s6_new(html_tree, contexto, selectors=selectors)
        productos_final.extend(productos_pagina)



🔎 Scrapeando categoría: Almacén/Aceites/Girasol
📄 Total de páginas: 1
➡️ Página 1: https://superseis.com.py/catalog/almacen/aceites/girasol?pageindex=1

🔎 Scrapeando categoría: Almacén/Aceites/Mezclas
📄 Total de páginas: 1
➡️ Página 1: https://superseis.com.py/catalog/almacen/aceites/mezclas?pageindex=1

🔎 Scrapeando categoría: Almacén/Aceites/Oliva
📄 Total de páginas: 1
➡️ Página 1: https://superseis.com.py/catalog/almacen/aceites/oliva?pageindex=1

🔎 Scrapeando categoría: Almacén/Aceites/Otros
📄 Total de páginas: 1
➡️ Página 1: https://superseis.com.py/catalog/almacen/aceites/otros?pageindex=1

🔎 Scrapeando categoría: Almacén/Aceites/Soja
📄 Total de páginas: 1
➡️ Página 1: https://superseis.com.py/catalog/almacen/aceites/soja?pageindex=1

🔎 Scrapeando categoría: Almacén/Aderezos_y_condimentos/Barbacoas
📄 Total de páginas: 1
➡️ Página 1: https://superseis.com.py/catalog/almacen/aderezos-y-condimentos/barbacoas?pageindex=1

🔎 Scrapeando categoría: Almacén/Aderezos_y_condimentos/Chimic

In [23]:
df = pd.DataFrame(productos_final)
df.head()

,titulo,precio,unidad_medida,marca,category_slug,ingestion_time,supermercado
0,"Aceite de girasol botella Alsamar 1,5 litros",₲ 28.700,unidad,,Almacén/Aceites/Girasol,2025-11-16 10:09:22.174644-03:00,Super Seis
1,Aceite de girasol botella Mirasol 500 ml,₲ 10.800,unidad,,Almacén/Aceites/Girasol,2025-11-16 10:09:22.174644-03:00,Super Seis
2,Aceite de girasol botella Mirasol 900 ml,₲ 17.500,unidad,,Almacén/Aceites/Girasol,2025-11-16 10:09:22.174644-03:00,Super Seis
3,Aceite de girasol botella Ok 900 ml,₲ 23.850,unidad,,Almacén/Aceites/Girasol,2025-11-16 10:09:22.174644-03:00,Super Seis
4,Aceite de girasol en bidón Natura 3 litros,₲ 82.500,unidad,,Almacén/Aceites/Girasol,2025-11-16 10:09:22.174644-03:00,Super Seis


In [24]:
save_df_as_csv(
    dataframe = df,
    name = 's6_productos',
    subfolder = 's6/productos'
)

[💾] Guardado en: /workspaces/tesis-ivan-gennaro/scripts/bronze/outputs/s6/productos/s6_productos_2025-11-16_11-01-53.csv
